# Indian Food Nutrition — Colab Training

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`.

Run cells top to bottom. The only things you need to fill in are:
- `GITHUB_REPO` in cell 2 (after you push to GitHub)
- Your `kaggle.json` upload in cell 4

Expected total time on T4: ~1.5–2 hours (25 epochs × ~4 min/epoch).

## Step 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run.")
print(result.stdout)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2 — Clone repo

Fill in your GitHub repo URL. If the repo is **public**, use `https://github.com/USERNAME/food-nutrition.git`.
If **private**, use `https://TOKEN@github.com/USERNAME/food-nutrition.git` (create a token at GitHub → Settings → Developer settings → Personal access tokens → Fine-grained, repo read access is enough).

In [ ]:
GITHUB_REPO = "https://github.com/Kaustubh-1420/food-nutrition.git"

import subprocess, os

if os.path.exists("food-nutrition"):
    print("Repo already cloned, pulling latest...")
    subprocess.run(["git", "-C", "food-nutrition", "pull"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO, "food-nutrition"], check=True)

os.chdir("food-nutrition")
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

## Step 3 — Install missing dependencies

Colab already has torch/torchvision/numpy/pillow — only installing what's missing.

In [ ]:
%%bash
pip install -q \
    timm==1.0.11 \
    onnx==1.17.0 \
    onnxsim==0.4.36 \
    onnxruntime==1.19.2 \
    scikit-learn==1.5.2
echo "Done."

## Step 4 — Kaggle credentials

Download your `kaggle.json` from [kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token.
Then run this cell and upload the file when prompted.

In [ ]:
import os, json

# Paste your Kaggle credentials here
KAGGLE_USERNAME = "your_kaggle_username"   # <-- fill in
KAGGLE_KEY      = "your_api_token"         # <-- fill in

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
creds_path = f"{kaggle_dir}/kaggle.json"
with open(creds_path, "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(creds_path, 0o600)
print("Kaggle credentials saved.")

## Step 5 — Download dataset

In [ ]:
%%bash
pip install -q kaggle
mkdir -p data/raw
kaggle datasets download -d l33tc0d3r/indian-food-classification -p data/raw --unzip
echo "Classes found: $(ls data/raw | wc -l)"
echo "Sample classes: $(ls data/raw | head -5)"
echo "Total images: $(find data/raw -type f | wc -l)"

## Step 6 — Generate train/val/test splits

In [ ]:
%%bash
python -m training.prepare_data --raw data/raw --out data/splits --seed 42
echo ""
echo "Split files:"
wc -l data/splits/*.csv

## Step 7 — Train EfficientNet-B2

25 epochs: 2 warmup (head only) + 23 full fine-tune with cosine LR decay.  
Best checkpoint saved to `models/checkpoints/best.pt` by val macro-F1.  
Target: ≥ 0.75 macro-F1, ≥ 0.80 top-1 accuracy.

In [ ]:
%%bash
python -m training.train \
    --splits data/splits \
    --out models/checkpoints \
    --epochs 25 \
    --batch-size 32 \
    --lr 3e-4 \
    --workers 2

## Step 8 — Export to ONNX

Simplifies the graph with `onnxsim` and verifies output parity (max diff < 1e-3).

In [ ]:
%%bash
python -m training.export_onnx \
    --ckpt models/checkpoints/best.pt \
    --out models/model.onnx
echo ""
ls -lh models/model.onnx

## Step 9 — Download model.onnx

This downloads `model.onnx` to your local machine. Copy it to `models/model.onnx` in your local repo, then push to HF Spaces.

In [ ]:
from google.colab import files
import os

onnx_path = "models/model.onnx"
if not os.path.exists(onnx_path):
    raise FileNotFoundError(f"{onnx_path} not found — did Step 8 complete successfully?")

size_mb = os.path.getsize(onnx_path) / 1e6
print(f"Downloading {onnx_path} ({size_mb:.1f} MB)...")
files.download(onnx_path)

## Step 10 — (Optional) Download training history

Useful for plotting loss/F1 curves.

In [ ]:
import json
import os

history_path = "models/checkpoints/history.json"
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print(f"{'Epoch':>5}  {'Phase':<20}  {'Loss':>8}  {'Val Acc':>8}  {'Val F1':>8}")
    print("-" * 60)
    for h in history:
        print(f"{h['epoch']:>5}  {h['phase']:<20}  {h['train_loss']:>8.4f}  {h['val_acc']:>8.4f}  {h['val_f1']:>8.4f}")
    
    from google.colab import files
    files.download(history_path)
else:
    print("No history.json found.")